In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
import os
import shutil
import tarfile
import zipfile
from pathlib import Path
from google.colab import drive

In [ ]:
#First mount Drive for the dataset if it is not mounted or session is restarted.
drive.mount('/content/drive')

In [ ]:
#Clone or pull the latest code
import os
REPO = "LaneDetection"
if not os.path.exists(f"/content/{REPO}"):
    !git clone https://github.com/abdullahtapanci/LaneDetection.git /content/{REPO}
else:
    !cd /content/{REPO} && git pull

%cd /content/{REPO}

EXTRACT THE DATA INTO COLAB ENVIRONMENT
1.SETUP

In [ ]:
# =========================
# CULane local extraction setup (Colab)
# =========================

# 1) Configure paths
DRIVE_CULANE_DIR = Path("/content/drive/MyDrive/CULane_Dataset")
LOCAL_WORK_DIR   = Path("/content")
LOCAL_ARCHIVE_DIR = LOCAL_WORK_DIR / "culane_archives"
LOCAL_CULANE_DIR  = LOCAL_WORK_DIR / "CULane"

LOCAL_ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CULANE_DIR.mkdir(parents=True, exist_ok=True)

# 2) Which archives to process
# Include only files that exist in your Drive folder
ARCHIVES = [
    "driver_100_30frame.tar.gz",
    "driver_161_90frame.tar.gz",
    "driver_182_30frame.tar.gz",
    "driver_193_90frame.tar.gz",
    "driver_37_30frame.tar.gz",
    "annotations_new.tar.gz",
    "laneseg_label_w16.tar.gz",
    "laneseg_label_w16_test.zip",
    "list.tar.gz",
]

# 3) Copy from Drive -> local (skip if already copied with same size)
def copy_if_needed(src: Path, dst: Path):
    if not src.exists():
        print(f"[WARN] Missing on Drive: {src.name}")
        return False
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        print(f"[SKIP] Already copied: {src.name}")
        return True
    print(f"[COPY] {src.name}")
    shutil.copy2(src, dst)
    return True

copied = []
for name in ARCHIVES:
    src = DRIVE_CULANE_DIR / name
    dst = LOCAL_ARCHIVE_DIR / name
    ok = copy_if_needed(src, dst)
    if ok:
        copied.append(dst)

print(f"\nTotal ready archives: {len(copied)}")

EXTRACTION PROCESS

In [ ]:
# =========================
# Extract archives into /content/CULane
# =========================
from pathlib import Path
import tarfile, zipfile

LOCAL_ARCHIVE_DIR = Path("/content/culane_archives")
LOCAL_CULANE_DIR  = Path("/content/CULane")
LOCAL_CULANE_DIR.mkdir(parents=True, exist_ok=True)

def extract_tar_gz(archive_path: Path, out_dir: Path):
    print(f"[EXTRACT tar] {archive_path.name}")
    with tarfile.open(archive_path, "r:gz") as tar:
        tar.extractall(path=out_dir)

def extract_zip(archive_path: Path, out_dir: Path):
    print(f"[EXTRACT zip] {archive_path.name}")
    with zipfile.ZipFile(archive_path, "r") as z:
        z.extractall(path=out_dir)

for p in sorted(LOCAL_ARCHIVE_DIR.iterdir()):
    if p.suffixes[-2:] == [".tar", ".gz"]:
        extract_tar_gz(p, LOCAL_CULANE_DIR)
    elif p.suffix == ".zip":
        extract_zip(p, LOCAL_CULANE_DIR)
    else:
        print(f"[SKIP] Unknown format: {p.name}")

print("\nExtraction complete.")

VERIFY

In [ ]:
# =========================
# Quick verification
# =========================
from pathlib import Path

root = Path("/content/CULane")

# count common file types
jpg_count = len(list(root.rglob("*.jpg")))
png_count = len(list(root.rglob("*.png")))
lines_count = len(list(root.rglob("*.lines.txt")))
list_files = list(root.rglob("*list*")) + list(root.rglob("train*.txt")) + list(root.rglob("val*.txt")) + list(root.rglob("test*.txt"))

print(f"Root: {root}")
print(f"JPG count      : {jpg_count}")
print(f"PNG count      : {png_count}")
print(f"lines.txt count: {lines_count}")
print(f"List-like files: {len(list_files)}")

# Show some list files for sanity
for f in sorted(set(list_files))[:20]:
    print(" -", f)